<a href="https://colab.research.google.com/github/PraneshKannan1203/Pranesz/blob/main/AI_Dungeon_Master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

!pip install -qU langchain langchain-google-genai langchain-community

import os
import sys
from typing import List, Dict

try:
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
    from langchain_core.runnables.history import RunnableWithMessageHistory
    from langchain_community.chat_message_histories import ChatMessageHistory

    from langchain_google_genai import ChatGoogleGenerativeAI
except ImportError as e:
    print(f"Error: Missing dependencies. {e}")
    print("Please install them using: pip install langchain langchain-google-genai langchain-community")
    sys.exit(1)

def get_api_key():
    """Securely get the API key from env or user input."""
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        print("\n=== Setup Required ===")
        print("This game requires a Google Gemini API Key.")
        print("You can get one for free at: https://aistudio.google.com/")
        api_key = input("Please enter your GOOGLE_API_KEY: ").strip()
    return api_key

class GameEngine:
    def __init__(self, api_key: str):
        self.llm = ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            google_api_key=api_key,
            temperature=0.7,
            convert_system_message_to_human=True
        )

        self.memory = ChatMessageHistory()

        self.system_prompt = """
        You are the Dungeon Master (DM) for an immersive text-based RPG called 'Echoes of Aethelgard'.

        GAME RULES:
        1. You describe the world, NPCs, and outcomes of actions vividly.
        2. You MUST track the player's 'Inventory' and 'Health' implicitly in the story.
        3. If the player tries to do something impossible, explain why they fail.
        4. If the player's health reaches zero, the game ends.
        5. Keep responses concise (under 4 sentences) unless it's a major plot point.
        6. Offer 2-3 suggested actions at the end of your description, but allow free typing.

        SETTING:
        A dark fantasy world where magic is fading. The player starts waking up in a cold, damp cell in an abandoned watchtower.

        START:
        Ask the player for their character's name and class (Warrior, Mage, or Rogue) to begin.
        """

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", self.system_prompt),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}")
        ])

        self.chain = self.prompt | self.llm

    def process_turn(self, user_input: str) -> str:
        """Sends user input to the LLM and gets the game response."""

        self.memory.add_user_message(user_input)

        response = self.chain.invoke({
            "input": user_input,
            "history": self.memory.messages
        })

        self.memory.add_ai_message(response.content)

        return response.content

def main():
    print("Welcome to 'Echoes of Aethelgard' - An AI Powered Text Adventure")
    print("---------------------------------------------------------------")

    api_key = get_api_key()
    game = GameEngine(api_key)

    print("\nInitialize Game World...\n")
    try:
        opening_scene = game.process_turn("Start the game. Describe where I am.")
        print(f"\nDM: {opening_scene}")

        while True:
            user_input = input("\n> ")

            if user_input.lower() in ['quit', 'exit']:
                print("Thanks for playing!")
                break

            if not user_input.strip():
                continue

            print("...")
            response = game.process_turn(user_input)


            print(f"\nDM: {response}")

    except Exception as e:
        print(f"\nAn error occurred: {e}")
        print("Check your API key or internet connection.")

if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.9.0 which is incompatible.
Welcome to 'Echoes of Aethelgard' - An AI Powered Text Adventure
-----------------------------------------------------